In [78]:
import numpy as np
import pandas as pd
import sns
from pandas.core.common import random_state
from scipy.stats import alpha
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

import training
import importlib
import mlflow
from sklearn.linear_model import LogisticRegression, LinearRegression
import dagshub
import ML_Assignment_1.preprocessor as pre


from ML_Assignment_1.mapping import ORDINAL_COLUMNS
from ML_Assignment_1.preprocessor import WOEEncoder

importlib.reload(training)
importlib.reload(pre)
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

In [79]:
dagshub.init(repo_owner='lukaLomadze', repo_name='ML_Assignment_1', mlflow=True)
mlflow.set_experiment("house-prices-finall")

Initialized MLflow to track repo "lukaLomadze/ML_Assignment_1"

Repository lukaLomadze/ML_Assignment_1 initialized!

<Experiment: artifact_location='mlflow-artifacts:/ecd0ff3db91b46ebb66568375b42439f', creation_time=1776172347248, experiment_id='4', last_update_time=1776172347248, lifecycle_stage='active', name='house-prices-finall', tags={}, workspace='default'>

In [80]:
train= pd.read_csv("../houses/train.csv")
test= pd.read_csv("../houses/test.csv")
test_ids=test["Id"]

In [81]:
missing = train.isna().mean()
high_missing = missing[missing>0.70].index.tolist()
is_outlier= (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)
clean_train = train.loc[~is_outlier].reset_index(drop=True)
train = clean_train.drop(columns=high_missing)
test = test.drop(columns=[c for c in high_missing if c in test.columns])


In [82]:
y = np.log1p(train['SalePrice'])
x=train.drop(columns=['SalePrice',"Id"])
xTest= test.drop(columns=['Id'])

In [83]:
print(y.shape, x.shape, xTest.shape)

(1458,) (1458, 75) (1459, 75)


In [84]:
from mapping import ORDINAL_COLUMNS
filler = pre.NAFiller()
qual = pre.QualityEncoder(ORDINAL_COLUMNS=ORDINAL_COLUMNS)
feat = pre.FeatureAdder()

x= feat.fit_transform(qual.fit_transform(filler.fit_transform(x,y)))
xTest = feat.transform(qual.transform(filler.transform(xTest)))

In [85]:
cat_cols= x.select_dtypes(include="object").columns
cardinal= x[cat_cols].nunique()
woe_cols = cardinal[cardinal>5].index.tolist()

C:\Users\lukas\AppData\Local\Temp\ipykernel_24024\1361995879.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols= x.select_dtypes(include="object").columns


In [86]:
woe_enc = pre.WOEEncoder(columns=woe_cols)
x=woe_enc.fit_transform(x,y)
xTest=woe_enc.transform(xTest)

In [87]:
ohe = pre.OneHotEncoderSafe()
x=ohe.fit_transform(x,y)
xTest=ohe.transform(xTest)


In [88]:
scaler = StandardScaler()
x = pd.DataFrame(scaler.fit_transform(x), columns=x.columns)
xTest = pd.DataFrame(scaler.transform(xTest), columns=x.columns)

In [89]:
from sklearn.feature_selection import RFE

rfe = RFE(estimator=Ridge(), n_features_to_select=50)
rfe = rfe.fit(x, y)
rfe_cols= x.columns[rfe.support_].tolist()
x= x[rfe_cols]
xTest = xTest[rfe_cols]


In [90]:
corr = x.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop= set()
for col in upper.columns:
    for p in upper.index[upper[col]>0.85].tolist():
        dcol = p if abs(x[col].corr(y))>= abs(x[p].corr(y)) else col
        drop.add(dcol)
corr_cols= [c for c in rfe_cols if c not in drop]
x = x[corr_cols]
xTest = xTest[corr_cols]


In [91]:
print("survuved features: ", x.columns)

survuved features:  Index(['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'BsmtQual',
       'BsmtExposure', 'BsmtFinSF1', 'TotalBsmtSF', 'HeatingQC', '1stFlrSF',
       '2ndFlrSF', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'GarageYrBlt', 'GarageCars',
       'GarageQual', 'WoodDeckSF', 'ScreenPorch', 'TotalSF', 'TotalPorchSF',
       'TotalBathrooms', 'HouseAge', 'RemodAge', 'HasPool', 'HasGarage',
       'Neighborhood_woe', 'Condition1_woe', 'Exterior1st_woe',
       'Foundation_woe', 'SaleCondition_woe', 'MSZoning_FV', 'MSZoning_RH',
       'MSZoning_RL', 'MSZoning_RM', 'LotConfig_CulDSac', 'LandSlope_Sev',
       'BldgType_Twnhs', 'MasVnrType_Stone', 'CentralAir_Y'],
      dtype='str')


In [92]:
import pickle
from mlflow.artifacts import download_artifacts


run_id= "6bebadfc353d4ce5a8629bcccdfb5549"




local_path = download_artifacts(artifact_uri=f"runs:/{run_id}/model.pkl")

print(local_path)

with open(local_path, "rb") as f:
    model = pickle.load(f)
print(type(model))




C:\Users\lukas\AppData\Local\Temp\tmp99tclm79\model.pkl
<class 'sklearn.linear_model._ridge.Ridge'>


In [93]:
model.fit(x,y)
pred= np.expm1(model.predict(xTest))
sub= pd.DataFrame({'Id': test_ids, 'SalePrice': pred})
sub.to_csv("submission.csv", index=False)
print("file saved")
print(sub)

file saved
        Id      SalePrice
0     1461  118128.399391
1     1462  159193.752394
2     1463  178753.898810
3     1464  198166.528103
4     1465  194465.264737
...    ...            ...
1454  2915   86491.816847
1455  2916   91347.350133
1456  2917  176201.255696
1457  2918  117295.701893
1458  2919  229330.087351

[1459 rows x 2 columns]
